# Demo 04 : Data and content pipelines

We rank the service tickets by mechanic hours, then turn sourced procedure details into a checked draft bulletin. Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · data-analysis

We load the service tickets into SQLite, rank mechanic hours by fault code in SQL, and ask the model to explain the returned rows. A check written in code compares every number in the explanation with the rows.

In [2]:
import csv
import re
import sqlite3
from pathlib import Path

db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE tickets (ticket_id TEXT, error_code TEXT, symptom TEXT, hours REAL)")
with open(data("demo-world/tickets.csv"), encoding="utf-8") as f:
    db.executemany("INSERT INTO tickets VALUES (?, ?, ?, ?)",
                   [(r["ticket_id"], r["error_code"], r["symptom"], float(r["hours"])) for r in csv.DictReader(f)])
db.execute("PRAGMA query_only = ON")

QUERY = ("SELECT error_code, COUNT(*) AS tickets, SUM(hours) AS hours FROM tickets "
         "GROUP BY error_code ORDER BY hours DESC LIMIT 5")
rows = db.execute(QUERY).fetchall()
print("query rows:", rows)
try:
    db.execute("DELETE FROM tickets")
except sqlite3.OperationalError as error:
    print("write rejected:", error)

query rows: [('F-RR-120', 7, 27.5), ('F-MM-210', 7, 16.0), ('F-RR-240', 5, 14.5), ('F-CD-101', 4, 12.0), ('F-CD-202', 4, 10.0)]
write rejected: attempt to write a readonly database


The model explains the rows in words. Every number it writes must come from the rows, so we extract the numbers and compare them in code.

In [3]:
explanation = llm.invoke("Explain this ranking of mechanic hours by fault code in two sentences. "
                         "Columns: error_code, tickets, hours. Use digits, and only numbers that appear in the rows. "
                         + repr(rows)).content
print("explanation:\n", explanation)

allowed = {float(value) for row in rows for value in row[1:]}
numbers = [float(n) for n in re.findall(r"\d+(?:\.\d+)?", re.sub(r"F-[A-Z]{2}-\d{3}", "", explanation))]
unsupported = [n for n in numbers if n not in allowed]
print("numbers in explanation:", numbers)
print("figure check:", "pass" if not unsupported else f"reject, not in the rows: {unsupported}")

explanation:
 F-RR-120 and F-MM-210 each had 7 tickets, but F-RR-120 required 27.5 mechanic hours compared with 16.0 for F-MM-210. Among the remaining codes, F-RR-240 had 5 tickets and 14.5 hours, while F-CD-101 and F-CD-202 each had 4 tickets and 12.0 and 10.0 hours, respectively.
numbers in explanation: [7.0, 27.5, 16.0, 5.0, 14.5, 4.0, 12.0, 10.0]
figure check: pass


## b · content-pipeline

We retrieve a local procedure, draft a service bulletin, and run deterministic checks before the draft can leave the pipeline.

In [4]:
procedure_path = Path(data_folder("demo-world/manuals")) / "cd-12-procedure-cd-af-01.md"
procedure = procedure_path.read_text(encoding="utf-8")
draft = llm.invoke(f"Draft a brief internal bulletin for F-CD-101 from this procedure. Cite the source file name {procedure_path.name}, include the part, the two-hour workshop estimate, and the post-repair self test. Do not add instructions. "
                   f"\nSource {procedure_path.name}:\n{procedure}").content
print("draft:\n", draft)

draft:
 **F-CD-101 — Internal bulletin**

For error F-CD-101 (engine intake pressure low), the procedure specifies replacing the engine air filter, part **CD-AF-01**. Allow **2 hours in the workshop**. After repair, run the self test; the error must not return.

Source: `cd-12-procedure-cd-af-01.md`


In [5]:
checks = {
    "source named": procedure_path.name in draft,
    "part named": "CD-AF-01" in draft,
    "time retained": "2 hour" in draft.lower() or "two hour" in draft.lower(),
    "self test retained": "self test" in draft.lower() or "self-test" in draft.lower(),
}
print("checks:", checks)
print("release:", "ready for editor" if all(checks.values()) else "hold for correction")

checks: {'source named': True, 'part named': True, 'time retained': True, 'self test retained': True}
release: ready for editor


The checks establish that required details appear in the draft. An editor still compares each claim with the retrieved procedure before publication.

## Things to try

- Change the SQL grouping from fault code to mechanic, and read the figure check
- Remove one required detail from the prompt and inspect the pipeline result